In [2]:
!pip install sentence-transformers chromadb groq pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [30]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

In [4]:
import os
print("All libraries imported sucessfully")
print("Ready to build a RAG system")

All libraries imported sucessfully
Ready to build a RAG system


In [36]:
GROQ_API_KEY="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"
os.environ["GROQ_API_KEY"]=GROQ_API_KEY
groq_client=Groq(api_key=GROQ_API_KEY)
print("Groq api client initialized")
print("Note:if you see an authentication error later,double check your api key")

Groq api client initialized
Note:if you see an authentication error later,double check your api key


In [7]:
df=pd.read_csv('college_notes.csv')
print("shape of dataset: ",df.shape)
print("\ncolumn names: ",df.columns.tolist())

print("\nFirst 3 rows: ")
print(df.head(3))

shape of dataset:  (15, 4)

column names:  ['note_id', 'subject', 'topic', 'content']

First 3 rows: 
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [8]:
print("subject in the dataset: ")
print(df['subject'].value_counts)
print("\nsample of topics: ")
print(df[['note_id','subject','topic']].to_string(index=False))
print("\nlength of content(number of characters) for each notes: ")
df['content_length']=df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))

subject in the dataset: 
<bound method IndexOpsMixin.value_counts of 0       Data Engineering
1       Data Engineering
2       Data Engineering
3       Data Engineering
4       Data Engineering
5       Machine Learning
6       Machine Learning
7       Machine Learning
8       Machine Learning
9       Machine Learning
10         Generative AI
11         Generative AI
12         Generative AI
13    Python Programming
14    Python Programming
Name: subject, dtype: object>

sample of topics: 
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feat

chunking

In [9]:
documents=df['content'].tolist()
ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]

metadatas=[
    {"subject":row['subject'],"topic":row['topic']}
    for row in df.to_dict('records')
]
print(f"total chunks prepared: {len(documents)}")
print(f'first document ID: {ids[0]}')
print(f'first document metadata: {metadatas[0]}')
print(f'first 100 chars of document: {documents[0][:100]}+"...')

total chunks prepared: 15
first document ID: note_N001
first document metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
first 100 chars of document: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc+"...


In [10]:
print("Loading embeddings model...")
print("this is may take 30-60 seconds on first run --model is being downloaded")
print("(subsequent runs will be faster as the model is cahed)")

embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded sucessfully")
test_embedding=embedding_model.encode("this is a test sentence")
print("test embedding shape: ",test_embedding.shape)
print("first 5 values of test document: ",test_embedding[:5])

Loading embeddings model...
this is may take 30-60 seconds on first run --model is being downloaded
(subsequent runs will be faster as the model is cahed)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded sucessfully
test embedding shape:  (384,)
first 5 values of test document:  [0.07155243 0.06848023 0.00660337 0.10176966 0.01112225]


In [11]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection('college_notes_rag')
print("chroma client created")
print("colection name:college_notes_rag")
print(f"documents in collection so far: {collection.count()}" )

chroma client created
colection name:college_notes_rag
documents in collection so far: 0


In [12]:
print("generating embeddings for all 15 notes")
print("this may take 15-30 seconds...")

embeddings=embedding_model.encode(documents,show_progress_bar=True)
print("embeddings matrix shape: ",embeddings.shape)
collection.add(
documents=documents,
embeddings=embeddings,
ids=ids,
metadatas=metadatas,
)
print(f"documents in collection now: {collection.count()}")

generating embeddings for all 15 notes
this may take 15-30 seconds...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings matrix shape:  (15, 384)
documents in collection now: 15


In [13]:
def retrieve_relevant_chunks(question,top_k=3):
  """Given a user question,retreive the most relevent document chunks from chromadb.
  parameters:
  question(str):the users's question as a text string
  top_k(int):How many top results to return (default:3)
  returns:
  a dictionary containing retrived documents,distances,metadata"""

  question_embedding=embedding_model.encode(question).tolist()
  results=collection.query(
     query_embeddings=question_embedding,
     n_results=top_k
  )
  return results
print("Retrivel function defined sucessfully")
print("functions retrive_relevent_chunks(metadata,top_k=3)")

Retrivel function defined sucessfully
functions retrive_relevent_chunks(metadata,top_k=3)


In [14]:
test_question="what is ETL and how does it work in data engineering"
print(f"test question:{test_question}")
print("="*60)

results =retrieve_relevant_chunks(test_question, top_k=3)

print("\ top 3 retrived chunks")
print("="*60)

for i,(doc,dist,meta) in enumerate(zip(results['documents'][0],results['distances'][0],results['metadatas'][0])):
  print(f"results:{i+1}")
  print(f"subject:{meta['subject']}")
  print(f"topic:{meta['topic']}")
  print(f"distance:{dist:.4f}")
  print(f"content:{doc[:120]}...")



test question:what is ETL and how does it work in data engineering
\ top 3 retrived chunks
results:1
subject:Data Engineering
topic:ETL Pipelines
distance:0.2041
content:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i...
results:2
subject:Data Engineering
topic:APIs and Data Collection
distance:1.1100
content:An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ...
results:3
subject:Python Programming
topic:Data Visualization
distance:1.3892
content:Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo...


<>:7: SyntaxWarning: invalid escape sequence '\ '
<>:7: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_1247/4079539311.py:7: SyntaxWarning: invalid escape sequence '\ '
  print("\ top 3 retrived chunks")


context injection:


the process of appending external data—like documents, user preferences, or session history—to a prompt so the AI can tailor its response

RAG Prompt template:

system->user->question->answer


In [15]:
def build_content_from_results(results):
  """format chromadb retrieval results into a readable context string.
  parameters:
  results:the output from collection.query()-a dictionary
  returns:
  context_str(str):a formated string of all retrieved documents"""

  context_parts=[]
  for i,(doc,meta) in enumerate(zip(results['documents'][0],results['metadatas'][0])):
    chunk_text = f"Source {i+1}: {meta['subject']} - {meta['topic']}\n{doc}"
    context_parts.append(chunk_text)

  context_str="\n\n---\n\n".join(context_parts)
  return context_str
context=build_content_from_results(results)
print("built context string from retrieved chunks")
print("="*60)
print(context[:500]+"...")
print("total context length: ",len(context),"characters")


built context string from retrieved chunks
Source 1: Data Engineering - ETL Pipelines
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.

---

Source 2: Data Engineering - APIs and Data Collection
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weat...
total context length:  850 characters


In [16]:
def generate_rag_answer(question, context):
    """
    Send the retrieved context and a question to the LLM
    and return the generated answer.
    """

In [17]:
def generate_rag_answer(question, context):
    system_prompt = """You are a helpful academic assistant for engineering students.

You will be given context retrieved from a college knowledge base and a student's question.

RULES:
1. Answer ONLY using the information provided in the context below.
2. If the answer is not found in the context, say exactly:
"I don't have enough information in my knowledge base to answer this question."
3. Do not use your general training knowledge.
4. Keep answers clear, accurate, and beginner-friendly.
5. Mention which source the information came from when possible.
"""

    user_prompt = f"""Context from Knowledge Base:
{context}

---
Student's Question: {question}

Please answer the question based only on the context provided above.
"""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=500
    )

    answer = response.choices[0].message.content
    return answer


print("RAG generation function defined")

RAG generation function defined


In [43]:
def ask_college_assistant(question, top_k=3, verbose=True):
    """
    Ask a question to the RAG-based college assistant.

    Parameters:
        question (str): User's question.
        top_k (int): Number of relevant chunks to retrieve.
        verbose (bool): Whether to display intermediate steps.

    Returns:
        str: Generated answer from the knowledge base.
    """

    # Step 1: Retrieve relevant chunks
    if verbose:
        print(f"Question: {question}")
        print("=" * 60)
        print("Step 1: Retrieving relevant documents...")

    results = retrieve_relevant_chunks(question, top_k=top_k)

    if verbose:
        print(f"Retrieved {len(results['documents'][0])} chunks from the knowledge base")

        for i, meta in enumerate(results["metadatas"][0]):
            subject = meta.get("subject", "Unknown")
            topic = meta.get("topic", "Unknown")
            print(f"{i+1}. {subject} - {topic}")

    # Step 2: Build context
    if verbose:
        print("\nStep 2: Building context string...")

    context = build_content_from_results(results)

    if verbose:
        print(f"Context built: {len(context)} characters")

    # Step 3: Generate answer
    if verbose:
        print("\nStep 3: Sending context to LLM for answer generation...")

    answer = generate_rag_answer(question, context)

    # Step 4: Display answer
    if verbose:
        print("\n" + "=" * 60)
        print("ANSWER")
        print("=" * 60)
        print(answer)
        print("=" * 60)

    return answer


# Pipeline ready
print("Complete RAG pipeline defined.")
print("Example:")
print('ask_college_assistant("What is inheritance in C++?", top_k=5)')

Complete RAG pipeline defined.
Example:
ask_college_assistant("What is inheritance in C++?", top_k=5)


In [44]:
question_1="What is ETL and what are its three main stages?"
answer_1=ask_college_assistant(question_1,top_k=3,verbose=True)

Question: What is ETL and what are its three main stages?
Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base
1. Data Engineering - ETL Pipelines
2. Generative AI - Retrieval Augmented Generation
3. Generative AI - Prompt Engineering

Step 2: Building context string...
Context built: 924 characters

Step 3: Sending context to LLM for answer generation...

ANSWER
According to Source 1: Data Engineering - ETL Pipelines, ETL stands for Extract Transform Load. 

The three main stages of ETL are:

1. Extract: This stage involves collecting raw data from different sources.
2. Transform: This stage involves transforming the raw data into a clean and structured format.
3. Load: This stage involves loading the transformed data into a database or data warehouse for analysis.


In [45]:
question_2="How do embeddings help in building search system?"
answer_2=ask_college_assistant(question_2,top_k=3,verbose=True)

Question: How do embeddings help in building search system?
Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base
1. Generative AI - Retrieval Augmented Generation
2. Generative AI - Large Language Models
3. Machine Learning - Feature Engineering

Step 2: Building context string...
Context built: 904 characters

Step 3: Sending context to LLM for answer generation...

ANSWER
I don't have enough information in my knowledge base to answer this question.


In [46]:
question_3="what is the population of tokya?"
print("testing with an out-of-scope question(not in college notes):")
answer_3=ask_college_assistant(question_3,top_k=3,verbose=True)

testing with an out-of-scope question(not in college notes):
Question: what is the population of tokya?
Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base
1. Generative AI - Large Language Models
2. Python Programming - Pandas Library
3. Data Engineering - Big Data and PySpark

Step 2: Building context string...
Context built: 878 characters

Step 3: Sending context to LLM for answer generation...

ANSWER
I don't have enough information in my knowledge base to answer this question.


In [49]:
def retrieve_by_subject(question, subject_filter, top_k=3):
    """
    Retrieve relevant chunks only from a specific subject.

    Parameters:
        question (str): User query
        subject_filter (str): Subject to filter by
        top_k (int): Number of results to retrieve

    Returns:
        dict: ChromaDB query results
    """

    question_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k,
        where={"subject": subject_filter}
    )

    return results
print("retrieving only from GenAI subject: ")
print("="*50)
filtered_results=retrieve_by_subject(
      question="how do LLMs generate text?",
      subject_filter="GenAI",
      top_k=3
  )
for i,(doc,meta) in enumerate(zip(filtered_results['documents'][0],filtered_results['metadatas'][0])):
  print(f"result:{i+1}:[{meta['subject']}] {meta['topic']}")
  print(f"{doc[:100]}...")

retrieving only from GenAI subject: 


#Practice Ouestions




Q1. What is hallucination in the context of LLMs?
Answer

Hallucination occurs when a Large Language Model (LLM) generates information that sounds correct but is actually false, inaccurate, or unsupported by facts.

Example

User: Who invented Python in 2020?

LLM: Python was invented by John Smith in 2020.

This answer is completely false because Python was created by Guido van Rossum in 1991.

Why it happens
Missing knowledge
Ambiguous questions
Model predicts likely words rather than verifying facts
How RAG helps

RAG retrieves relevant documents before generating answers, reducing hallucinations.


Q2. What does RAG stand for? What problem does it solve?

Answer:

RAG = Retrieval-Augmented Generation

It retrieves relevant information from a database before generating an answer.

Problem Solved:

Reduces hallucinations
Gives more accurate answers

Q3. What is the role of a vector database in the RAG pipeline?

Answer:

A vector database stores document embeddings and finds the most relevant documents for a user's question.

Example:
ChromaDB stores note embeddings and retrieves similar notes when a question is asked.

Q4. What is the difference between the Indexing phase and the Querying phase of RAG?

Answer:

Indexing Phase
Documents are collected
Converted into embeddings
Stored in vector database
Querying Phase
User asks a question
Question is converted into embedding
Similar documents are retrieved
LLM generates the answer

Q5. Why must you use the same embedding model for both documents and queries?

Answer:

Because both documents and questions must be represented in the same vector space.

Using different embedding models can give inaccurate search results.

Q6. Why is a low temperature (e.g., 0.1) preferred for RAG-based LLM calls?

Answer:

Low temperature makes the model:

More accurate
More consistent
Less random
Less likely to hallucinate

Therefore, RAG systems usually use a low temperature like 0.1.

In [50]:
#Q7. Modify ask_college_assistant() to display distance scores.

#Answer:

#Distance scores show how similar each retrieved chunk is to the user's question.
for meta, distance in zip(results["metadatas"][0], results["distances"][0]):
    print(meta["topic"], "Distance:", distance)

ETL Pipelines Distance: 0.20405080914497375
APIs and Data Collection Distance: 1.1100008487701416
Data Visualization Distance: 1.389223337173462


In [52]:
#Q8. Change generate_rag_answer() so the LLM always responds in bullet points.
def generate_rag_answer(question, context):

    system_prompt = """
    You are a helpful academic assistant for engineering students.

    You will be given context retrieved from a college knowledge base and a student's question.

    RULES:
    1. Answer ONLY using the information provided in the context below.
    2. If the answer is not found in the context, say exactly:
       "I don't have enough information in my knowledge base to answer this question."
    3. Do not use your general training knowledge.
    4. Keep answers clear, accurate, and beginner-friendly.
    5. Mention which source the information came from when possible.
    6. ALWAYS format your answer using bullet points.
    """

    user_prompt = f"""
    Context from Knowledge Base:
    {context}

    ---
    Student's Question: {question}

    Please answer the question based only on the context provided above.
    """

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=500
    )

    answer = response.choices[0].message.content
    return answer

In [53]:
#Q9. Add a function that returns only the topic names of retrieved chunks.
def get_retrieved_topics(question, top_k=3):
    results = retrieve_relevant_chunks(question, top_k)

    topics = []

    for meta in results["metadatas"][0]:
        topics.append(meta["topic"])

    return topics